# Fetching country pres/abs for all species from GLOBI dataset

In [1]:
# ==============================================================================
# GBIF PRESENCE ENRICHMENT — FULL GRAPH
#
# Fetches yes/no country presence for every resolvable species in the graph
# (all SINAS invasives + all GloBI interaction partners), then builds:
#   - presence      : (num_species, num_countries) bool tensor
#   - pos_opp_mat   : (num_species, num_countries) float16 tensor
#                     fraction of sp's POSITIVE-valence GloBI partners
#                     (resource/mutualism — food, hosts, pollinators, symbionts)
#                     confirmed present in co
#   - neg_opp_mat   : (num_species, num_countries) float16 tensor
#                     fraction of sp's NEGATIVE-valence GloBI partners
#                     (enemy/competition — predators, parasites, pathogens,
#                     competitors) confirmed present in co
#                     Split by interaction category (Section 3b below) so a
#                     species surrounded by predators doesn't score identically
#                     to one surrounded by prey/mutualists.
#
# Run once overnight on Colab (Google-to-GBIF latency ~5ms vs ~80ms local).
# Saves to Google Drive so it survives session termination.
# Resumes automatically from checkpoint if interrupted.
#
# !pip install aiohttp nest_asyncio country_converter tqdm -q
# ==============================================================================

import asyncio, json, logging, random, time
from collections import defaultdict
from pathlib import Path

import aiohttp
import pandas as pd
import torch
from tqdm.auto import tqdm
import country_converter as coco
import nest_asyncio

logging.getLogger('country_converter').setLevel(logging.ERROR)

c:\Users\simon\Documents\GitHub\horizon-scanner\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# ==============================================================================
# CONFIG — adjust these to taste
# ==============================================================================
repo_root   = Path.cwd().parent # get repo root

sinas_path        = repo_root / 'data' / 'sinas_matched_species.csv'
network_path      = repo_root / 'data' / 'matched_globi_network.csv'
higher_order_path = repo_root / 'data' / 'matched_globi_network_higher_order.csv'
output_path       = repo_root / 'data' / 'gbif_presence.pt'
checkpoint_path   = repo_root / 'data' / 'gbif_presence_checkpoint.json'  # resume support

YOUR_EMAIL      = 'simon.reynaert@plantentuinmeise.be'  # ← tells GBIF who you are; gets better rate limits
MAX_CONCURRENT  = 50     # 10 workers → ~10 sp/s steady, no 429s
REQUEST_TIMEOUT = 30     # seconds per request before timeout
FACET_LIMIT     = 300    # covers all ~250 GBIF country codes
MAX_RETRIES     = 3      # per-request retry attempts
CHECKPOINT_EVERY = 200   # write checkpoint every N species
country_col     = 'location'


In [3]:

# ==============================================================================
# 1. REBUILD INDEX MAPPINGS
# ==============================================================================
print("📖 Rebuilding index mappings...")

df_sinas  = pd.read_csv(sinas_path,        engine='python', on_bad_lines='warn')
df_net    = pd.read_csv(network_path,       engine='python', on_bad_lines='warn')
df_higher = pd.read_csv(higher_order_path,  engine='python', on_bad_lines='warn')

df_sinas = df_sinas.dropna(subset=['gbif_id']).copy()
df_sinas['gbif_id'] = df_sinas['gbif_id'].astype(int)

sinas_sp   = set(df_sinas['gbif_id'].unique())
net_src_sp = set(df_net['source_gbif_id'].dropna().astype(int).unique())
net_tgt_sp = set(df_net['target_gbif_id'].dropna().astype(int).unique())
all_species_ids   = sorted(sinas_sp | net_src_sp | net_tgt_sp)
all_country_names = sorted(df_sinas[country_col].dropna().unique().tolist())

sp_to_idx = {sp_id: idx for idx, sp_id in enumerate(all_species_ids)}
co_to_idx = {name:  idx for idx, name  in enumerate(all_country_names)}

num_species   = len(all_species_ids)
num_countries = len(all_country_names)

# Build id_to_name for resolvability check
id_to_name = (df_sinas.drop_duplicates('gbif_id')
                       .set_index('gbif_id')['gbif_canonical_name']
                       .to_dict())
for row in df_net.dropna(subset=['source_gbif_id']).drop_duplicates('source_gbif_id').itertuples():
    if int(row.source_gbif_id) not in id_to_name:
        id_to_name[int(row.source_gbif_id)] = row.source_taxon_name
for row in df_net.dropna(subset=['target_gbif_id']).drop_duplicates('target_gbif_id').itertuples():
    if int(row.target_gbif_id) not in id_to_name:
        id_to_name[int(row.target_gbif_id)] = row.target_taxon_name

# Resolvable = has a real GBIF backbone name (not a placeholder ID)
def is_resolvable(gbif_id):
    name = str(id_to_name.get(gbif_id, ''))
    return not any(name.startswith(p) for p in ('GBIF:', 'Unresolved', 'nan'))

resolvable = {
    gid: sp_to_idx[gid]
    for gid in all_species_ids
    if is_resolvable(gid)
}

print(f"   Total species in graph : {num_species:,}")
print(f"   Resolvable (will fetch): {len(resolvable):,}")
print(f"   Unresolvable (skipped) : {num_species - len(resolvable):,}  "
      f"← placeholder GBIF: IDs from unmatched GloBI records")
print(f"   Countries              : {num_countries:,}")


📖 Rebuilding index mappings...
   Total species in graph : 167,357
   Resolvable (will fetch): 167,357
   Unresolvable (skipped) : 0  ← placeholder GBIF: IDs from unmatched GloBI records
   Countries              : 289


In [4]:
# ==============================================================================
# 1b. INTERACTION TYPE ONTOLOGY
#    Maps raw GloBI strings to ecological categories, and each category to a
#    valence (positive/negative) for the SOURCE species of the GloBI record.
#    Kept identical to the ontology in horizon_scanner_final_GPUsplit.ipynb
#    Section 2 — if you edit the term sets, edit both places together.
#
#    Valence is directional, not symmetric:
#      resource  (source consumes/uses target) -> positive for source
#                                                  (food available)
#                                               -> negative for target
#                                                  (source is its predator/parasite)
#      enemy     (source is preyed on/parasitized/infected BY target) -> negative
#                for source, positive for target (mirror image of 'resource')
#      mutualism -> positive for BOTH source and target (pollinators, symbionts,
#                   commensals — presence of either helps the other)
#      competition -> negative for BOTH source and target
#      unknown   -> contributes to neither pos nor neg opportunity
# ==============================================================================
CONSUMPTION_TERMS = {
    'eats','preysOn','parasitizes','pathogenOf','parasitoidOf','consumes',
    'endoparasitoidOf','ectoparasitoidOf','endoparasiteOf','ectoparasiteOf',
    'scavenges','browses','grazesOn','hasHost','kills','saprophyteOf',
    'hyperparasiteOf','hyperparasitoidOf','bloodFeedsOn','nectarFeedsOn',
    'pollenFeedsOn','seedEaterOf','woodBorerOf','leafMinerOf','gallMakerOf',
    'defoliatorOf','rootFeederOf','vectorOf', 'parasiteOf', 'hemiparasiteOf', 'ectoParasitoid',
    'kleptoparasiteOf', 'rootparasiteOf', 'laysEggsOn', 'laysEggsIn'
}

MUTUALISM_TERMS = {
    'pollinates','visitsFlowersOf','symbiontOf','mutualistOf','associatedWith',
    'coOccursWith','hasSymbiont','commensalistOf','hasCommensalist','epiphyteOf',
    'hasEpiphyte','inquilineOf','hasInquiline','phoreticOf','hasPhoretic',
    'mycorrhizalWith','dispersesSeedsOf','disperses','vectorFor','sheltersWithin',
    'providesShelterTo','nestedIn','hostsNestedIn','flowersVisitedBy','visitedBy', 'hasVector',
    'hasDispersalVector', 'guestOf', 'coRoostsWith', 'hasHabitat', 'hasRoost', 'livesOn',
    'livesInsideOf', 'livesUnder', 'livesNear', 'inhabits'
}
COMPETITION_TERMS = {
    'competesWith','interferesWith','allelopathicTo','inhibits',
    'displaces','antagonistOf', 'allelopathOf'
}
VICTIM_TERMS = {
    'hostOf','hasParasite','hasPathogen','hasParasitoid','preyedOnBy','eatenBy', 'providesNutrientsFor',
    'parasitizedBy','infectedBy','killedBy','preyedUponBy','parasitoidBy','pathogenBy'
}

def categorize_interaction(itype: str) -> str:
    """Map a raw GloBI interaction type to resource/enemy/mutualism/competition/unknown."""
    if pd.isna(itype): return 'unknown'
    c = itype.strip(); l = c.lower()
    if c in CONSUMPTION_TERMS:  return 'resource'
    if c in MUTUALISM_TERMS:    return 'mutualism'
    if c in COMPETITION_TERMS:  return 'competition'
    if c in VICTIM_TERMS:       return 'enemy'
    if 'by' in l and any(w in l for w in ['eat','prey','parasit','kill','infect','attack']):
        return 'enemy'
    if any(w in l for w in ['eat','prey','consume','host','feed','graze','browse','destroy']):
        return 'resource'
    if any(w in l for w in ['symbio','mutual','pollin','cooccur','visit','associat','commens','shelter']):
        return 'mutualism'
    if any(w in l for w in ['compet','inhib','displac','antagon','resist','exclude']):
        return 'competition'
    return 'unknown'

# category -> (valence for source, valence for target); +1 = positive, -1 = negative, 0 = neither
CATEGORY_VALENCE = {
    'resource':    (+1, -1),
    'enemy':       (-1, +1),
    'mutualism':   (+1, +1),
    'competition': (-1, -1),
    'unknown':     ( 0,  0),
}

all_interaction_types = pd.concat(
    [df_net['interaction_type'], df_higher['interaction_type']]
).dropna().unique()
interaction_mapping = {t: categorize_interaction(t) for t in all_interaction_types}

_cat_counts = pd.Series(list(interaction_mapping.values())).value_counts()
print("   Interaction category breakdown (unique raw terms):")
for cat, cnt in _cat_counts.items():
    print(f"     {cat}: {cnt}")
_all_mapped = (pd.concat([df_net['interaction_type'], df_higher['interaction_type']])
               .map(lambda t: interaction_mapping.get(t, 'unknown') if pd.notna(t) else 'unknown'))
_unk_frac = (_all_mapped == 'unknown').mean()
print(f"   {_unk_frac:.1%} of all interaction records are 'unknown' (excluded from pos/neg opportunity)")


   Interaction category breakdown (unique raw terms):
     mutualism: 19
     resource: 18
     unknown: 3
     competition: 1
     enemy: 1
   24.3% of all interaction records are 'unknown' (excluded from pos/neg opportunity)


In [5]:
# ==============================================================================
# 2. ISO2 → COUNTRY INDEX MAPPING
# ==============================================================================
SUB_NATIONAL_TO_ISO2 = {
    'Aegean':'GR','Caspian Sea':'RU','Alaska':'US','Hawaii':'US',
    'Virgin Islands (U.S.)':'US','United States Minor Outlying Islands':'US',
    'Puerto Rico':'US','Guam':'GU','American Samoa':'AS',
    'Northern Mariana Islands':'MP','Canary Islands':'ES','Balearic Islands':'ES',
    'Azores':'PT','Madeira':'PT','Sicily':'IT','Sardinia':'IT',
    'Tasmania':'AU','Lord Howe Islands':'AU','Norfolk Island':'NF',
    'Christmas Island':'CX','Cocos Islands':'CC','Galapagos':'EC',
    'Corsica':'FR','Réunion':'RE','Mayotte':'YT','Guadeloupe':'GP',
    'Martinique':'MQ','French Guiana':'GF','Saint Pierre and Miquelon':'PM',
    'Saint-Barthélemy':'BL','Saint-Martin':'MF','Shetland Islands':'GB',
    'Guernsey':'GG','Jersey':'JE','Isle of Man':'IM','Gibraltar':'GI',
    'Bermuda':'BM','Anguilla':'AI','Cayman Islands':'KY','Montserrat':'MS',
    'Turks and Caicos Islands':'TC','Virgin Islands (British)':'VG',
    'Falkland Islands':'FK','Pitcairn Islands':'PN','Saint Helena':'SH',
    'Vancouver Island':'CA','Crete':'GR','Zanzibar Island':'TZ',
    'Rapa Nui':'CL','Socotra Island':'YE','Nicobar and Andaman Islands':'IN',
    'Rodriguez Island':'MU','Hong Kong':'HK','Macao':'MO',
    'Faroe Islands':'FO','Greenland':'GL','Svalbard and Jan Mayen':'SJ',
    'Åland':'AX','Tokelau':'TK','Niue':'NU','Cook Islands':'CK',
    'Kermadec Islands':'NZ','Northern Cyprus':'CY','Western Sahara':'EH',
    'Fernando de Noronha':'BR','Palestine':'PS','Kosovo':'XK',
    'Antipodes Island':'NZ','Izu Islands':'JP','Ogasawara Islands':'JP',
}

cc = coco.CountryConverter()
iso2_to_co_indices = defaultdict(list)
for name in all_country_names:
    co_idx = co_to_idx[name]
    iso2   = SUB_NATIONAL_TO_ISO2.get(name) or cc.convert(names=name, to='ISO2')
    if iso2 and iso2 != 'not_found':
        iso2_to_co_indices[iso2.upper()].append(co_idx)

print(f"   ISO2 codes mapped: {len(iso2_to_co_indices):,}")


   ISO2 codes mapped: 246


In [6]:
# ==============================================================================
# 3. SEED PRESENCE MATRIX FROM SINAS
# ==============================================================================
presence = torch.zeros(num_species, num_countries, dtype=torch.bool)

df_sinas['sp_idx'] = df_sinas['gbif_id'].map(sp_to_idx)
df_sinas['co_idx'] = df_sinas[country_col].map(co_to_idx)
seed_df = df_sinas.dropna(subset=['sp_idx', 'co_idx'])
presence[seed_df['sp_idx'].astype(int).values,
         seed_df['co_idx'].astype(int).values] = True
n_seeded = presence.sum().item()
print(f"   Seeded {n_seeded:,} presences from SINAS")



   Seeded 356,506 presences from SINAS


In [17]:
# ==============================================================================
# 4. LOAD CHECKPOINT (resume support)
# ==============================================================================
completed  = {}
failed_ids = []

if checkpoint_path.exists():
    try:
        with open(checkpoint_path) as f:
            ckpt = json.load(f)
        completed  = {int(k): v for k, v in ckpt.get('completed', {}).items()}
        failed_ids = ckpt.get('failed', [])

        # Replay already-fetched data into presence matrix
        for gbif_id, co_indices in completed.items():
            sp_idx = resolvable.get(gbif_id)
            if sp_idx is not None and co_indices:
                presence[sp_idx,
                         torch.tensor(co_indices, dtype=torch.long)] = True

        print(f"\n♻️  Resuming from checkpoint:")
        print(f"   Already completed: {len(completed):,} species")
        print(f"   Previously failed: {len(failed_ids):,} species")
    except json.JSONDecodeError:
        print("⚠️  Corrupt checkpoint (interrupted write) — starting fresh.")
        checkpoint_path.unlink()

remaining = {
    gid: sidx
    for gid, sidx in resolvable.items()
    if gid not in completed
}
print(f"   Remaining to fetch: {len(remaining):,} species")


♻️  Resuming from checkpoint:
   Already completed: 167,357 species
   Previously failed: 251,638 species
   Remaining to fetch: 0 species


In [19]:
# ==============================================================================
# 5. ASYNC FETCHER
# ==============================================================================
# limit=0 handles occurrence queries instantly by requesting counts only
GBIF_FACET_URL = (
    "https://api.gbif.org/v1/occurrence/search"
    "?taxonKey={taxon_key}"
    "&limit=0"
    "&facet=country"
    f"&facetLimit={FACET_LIMIT}"
)

async def fetch_one(session, gbif_id, semaphore):
    url = GBIF_FACET_URL.format(taxon_key=gbif_id)
    for attempt in range(MAX_RETRIES):
        try:
            async with semaphore:
                async with session.get(
                    url,
                    timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
                ) as resp:
                    if resp.status == 429:
                        wait = 2 ** attempt * 10 + random.uniform(1, 5)
                        await asyncio.sleep(wait)
                        continue
                    if resp.status != 200:
                        await asyncio.sleep(2.0 * (attempt + 1) + random.uniform(0, 1))
                        continue
                    data = await resp.json(content_type=None)

            co_indices = []
            facets = data.get('facets', [])
            if facets:
                for entry in facets[0].get('counts', []):
                    iso2  = entry.get('name', '').upper()
                    count = entry.get('count', 0)
                    if count > 0 and iso2 in iso2_to_co_indices:
                        co_indices.extend(iso2_to_co_indices[iso2])
            return gbif_id, list(set(co_indices))

        except asyncio.TimeoutError:
            await asyncio.sleep(2.0 * (attempt + 1) + random.uniform(0, 1))
        except Exception:
            await asyncio.sleep(1.0 * (attempt + 1) + random.uniform(0, 1))

    return gbif_id, None


async def run_enrichment(fetch_dict):
    gbif_ids  = list(fetch_dict.keys())
    total     = len(gbif_ids)
    in_queue  = asyncio.Queue()
    out_queue = asyncio.Queue()

    for gid in gbif_ids:
        await in_queue.put(gid)

    semaphore     = asyncio.Semaphore(MAX_CONCURRENT)
    new_completed = {}
    new_failed    = []

    headers = {
        'User-Agent': f'HorizonScanner/1.0 (mailto:{YOUR_EMAIL})',
    }
    connector = aiohttp.TCPConnector(
        limit=MAX_CONCURRENT + 4,
        ttl_dns_cache=300,
        enable_cleanup_closed=True,
    )

    async def worker():
        while True:
            try:
                gid = in_queue.get_nowait()
            except asyncio.QueueEmpty:
                return
            result = await fetch_one(session, gid, semaphore)
            await out_queue.put(result)
            in_queue.task_done()

    async with aiohttp.ClientSession(connector=connector, headers=headers) as session:
        workers = [asyncio.create_task(worker()) for _ in range(MAX_CONCURRENT)]
        processed = 0
        pbar = tqdm(total=total, desc="GBIF fetch", unit="sp", dynamic_ncols=True)
        
        try:
            while processed < total:
                gbif_id, co_indices = await out_queue.get()
                processed += 1

                if co_indices is None:
                    new_failed.append(int(gbif_id))
                else:
                    new_completed[int(gbif_id)] = [int(x) for x in co_indices]
                    sp_idx = fetch_dict[gbif_id]
                    if co_indices:
                        presence[sp_idx, torch.tensor(co_indices, dtype=torch.long)] = True

                pbar.update(1)
                pbar.set_postfix({
                    'ok':     len(new_completed),
                    'failed': len(new_failed),
                }, refresh=False)

                # Windows-hardened Atomic Checkpoint
                if processed % CHECKPOINT_EVERY == 0 or processed == total:
                    all_completed = {
                        int(k): [int(x) for x in v]
                        for k, v in {**completed, **new_completed}.items()
                    }
                    all_failed = [int(x) for x in failed_ids + new_failed]
                    tmp_path = checkpoint_path.with_suffix('.tmp')
                    
                    try:
                        with open(tmp_path, 'w') as f:
                            json.dump({'completed': all_completed, 'failed': all_failed}, f)
                        
                        # Defend against aggressive OS/OneDrive locks
                        for lock_attempt in range(5):
                            try:
                                if checkpoint_path.exists():
                                    checkpoint_path.unlink()
                                tmp_path.rename(checkpoint_path)
                                break
                            except PermissionError:
                                if lock_attempt == 4:
                                    print(f"\n⚠️ Checkpoint write locked by Windows. Skipping this flush, but keeping data in memory...")
                                else:
                                    await asyncio.sleep(0.5)
                    except Exception as ce:
                        print(f"\n⚠️ Error updating checkpoint: {ce}")

        except (asyncio.CancelledError, KeyboardInterrupt):
            print("\n🛑 Fetching interrupted! Halting workers cleanly...")
            raise
        finally:
            # Fixes stuck tickers: force the widget to drop structural connections
            pbar.close()
            
            # Fixes zombie background loops: kill active connections instantly
            for w in workers:
                w.cancel()
            await asyncio.gather(*workers, return_exceptions=True)
            print("🧹 Active connections evicted. Loop is stable.")

    return new_completed, new_failed

In [20]:
# ==============================================================================
# 6. RUN MAIN FETCH
# ==============================================================================
nest_asyncio.apply()
loop = asyncio.get_event_loop()

if remaining:
    eta_min = len(remaining) // max(MAX_CONCURRENT * 6, 1)
    print(f"\n🌐 Fetching {len(remaining):,} species ({MAX_CONCURRENT} workers)...")

    new_completed, new_failed = loop.run_until_complete(run_enrichment(remaining))
    print(f"\n   Fetched : {len(new_completed):,}")
    print(f"   Failed  : {len(new_failed):,}")
else:
    print("\n✅ All species already accounted for — processing from database.")
    new_completed = {}
    new_failed    = []


✅ All species already accounted for — processing from database.


In [34]:

# ==============================================================================
# 7. AUTOMATIC RETRY PASSES FOR FAILED SPECIES
#    Up to 3 passes with increasing backoff. Most failures are transient
#    (timeout spike, momentary 503) and recover on the first retry pass.
# ==============================================================================
# Remove any IDs that eventually succeeded — failed_ids from the checkpoint
# may include species that were retried and recovered in a previous session.
# We only want species genuinely unresolved after the main fetch completes.
all_succeeded = set(completed.keys()) | set(int(k) for k in new_completed.keys())
all_failed    = [int(x) for x in failed_ids + new_failed
                 if int(x) not in all_succeeded]
 
if all_failed:
    print(f"\n🔁 Retrying {len(all_failed):,} failed species "
          f"(up to 3 passes with backoff)...")
 
    for retry_pass in range(1, 4):
        if not all_failed:
            break
 
        backoff = 30 * retry_pass
        print(f"\n   Pass {retry_pass}/3 — waiting {backoff}s then retrying "
              f"{len(all_failed):,} species...")
        time.sleep(backoff)
 
        retry_dict = {
            gid: resolvable[gid]
            for gid in all_failed
            if gid in resolvable
        }
        retry_completed, still_failed = loop.run_until_complete(
            run_enrichment(retry_dict)
        )
 
        new_completed.update(retry_completed)
        for gbif_id, co_indices in retry_completed.items():
            sp_idx = resolvable.get(gbif_id)
            if sp_idx is not None and co_indices:
                presence[sp_idx,
                         torch.tensor(co_indices, dtype=torch.long)] = True
 
        print(f"   Pass {retry_pass} recovered : {len(retry_completed):,} | "
              f"Still failing: {len(still_failed):,}")
        all_failed = [int(x) for x in still_failed]
 
    if all_failed:
        print(f"\n   ⚠️  {len(all_failed):,} species failed all retry passes.")
        print(f"   These will have SINAS-only presence data (or zeros if GloBI-only).")
    else:
        print(f"\n   ✅ All failed species recovered.")
else:
    all_failed = []

In [22]:
# ==============================================================================
# 8. BIOTIC OPPORTUNITY MATRICES (positive- and negative-valence, split)
#    pos_opp_mat[sp, co] = fraction of sp's POSITIVE-valence GloBI partners
#                          present in co (resource/mutualism: food, hosts,
#                          pollinators, symbionts -> partner presence helps
#                          sp establish)
#    neg_opp_mat[sp, co] = fraction of sp's NEGATIVE-valence GloBI partners
#                          present in co (enemy/competition: predators,
#                          parasites, pathogens, competitors -> partner
#                          presence hinders sp establishing)
#    Precomputed here so training-time lookup is a single tensor index op.
#
#    Valence is directional (see Section 1b): if source "eats" target, the
#    target is a resource for the source (positive for source) AND the
#    source is an enemy of the target (negative for target). Mutualism and
#    competition are symmetric — positive/negative for both sides.
#
#    If the output file already exists (from a previous run), we load
#    pos_opp_mat / neg_opp_mat directly from it rather than recomputing —
#    this saves significant time when the script is resumed after
#    multi-day fetching.
# ==============================================================================
if output_path.exists():
    print("\n⚙️ Loading pos_opp_mat / neg_opp_mat from existing output file (skipping recompute)...")
    _existing    = torch.load(output_path, map_location='cpu', weights_only=False)
    pos_opp_mat  = _existing['pos_opp_mat']   # (S, C) float16
    neg_opp_mat  = _existing['neg_opp_mat']   # (S, C) float16
    del _existing
    print(f"   Loaded pos_opp_mat: {list(pos_opp_mat.shape)} | neg_opp_mat: {list(neg_opp_mat.shape)}")
else:
    print("\n⚙️ Precomputing biotic opportunity matrices (pos/neg split)...")

    # Build sp_idx-keyed directed adjacency from GloBI edges, split by valence.
    # Unlike a plain undirected partner set, source/target roles matter here:
    # 'resource' (source eats target) contributes target -> source's positive
    # partners AND source -> target's negative partners; 'enemy' is the mirror
    # image; 'mutualism'/'competition' are symmetric.
    pos_partners = defaultdict(set)
    neg_partners = defaultdict(set)
    net_clean = df_net.dropna(subset=['source_gbif_id', 'target_gbif_id']).copy()
    net_clean['source_gbif_id'] = net_clean['source_gbif_id'].astype(int)
    net_clean['target_gbif_id'] = net_clean['target_gbif_id'].astype(int)
    n_skipped_unknown = 0
    for row in net_clean.itertuples():
        src_idx = sp_to_idx.get(row.source_gbif_id)
        tgt_idx = sp_to_idx.get(row.target_gbif_id)
        if src_idx is None or tgt_idx is None:
            continue
        category = interaction_mapping.get(row.interaction_type, 'unknown')
        src_valence, tgt_valence = CATEGORY_VALENCE.get(category, (0, 0))
        if src_valence == 0 and tgt_valence == 0:
            n_skipped_unknown += 1
            continue
        if src_valence > 0:
            pos_partners[src_idx].add(tgt_idx)
        elif src_valence < 0:
            neg_partners[src_idx].add(tgt_idx)
        if tgt_valence > 0:
            pos_partners[tgt_idx].add(src_idx)
        elif tgt_valence < 0:
            neg_partners[tgt_idx].add(src_idx)

    print(f"   Species with positive-valence partners: {len(pos_partners):,}")
    print(f"   Species with negative-valence partners: {len(neg_partners):,}")
    print(f"   Interaction records skipped ('unknown' category): {n_skipped_unknown:,}")

    # For each species, mean presence of its (valence-specific) partners
    # across all countries. float16 is sufficient precision for a [0,1]
    # fraction and halves memory.
    pos_opp_mat = torch.zeros(num_species, num_countries, dtype=torch.float16)
    neg_opp_mat = torch.zeros(num_species, num_countries, dtype=torch.float16)
    presence_f  = presence.float()   # temporary float32 view for mean computation

    CHUNK = 500

    def fill_opp_mat(mat, partner_dict, desc):
        sp_ids = list(partner_dict.keys())
        for i in tqdm(range(0, len(sp_ids), CHUNK), desc=desc):
            for sp in sp_ids[i: i + CHUNK]:
                partners = torch.tensor(list(partner_dict[sp]), dtype=torch.long)
                mat[sp] = presence_f[partners].mean(dim=0).half()

    fill_opp_mat(pos_opp_mat, pos_partners, "Positive biotic opportunity")
    fill_opp_mat(neg_opp_mat, neg_partners, "Negative biotic opportunity")

    del presence_f   # free the temporary float32 copy


⚙️ Precomputing biotic opportunity matrices (pos/neg split)...
   Species with positive-valence partners: 118,620
   Species with negative-valence partners: 43,579
   Interaction records skipped ('unknown' category): 220,045


Positive biotic opportunity:   0%|          | 0/238 [00:00<?, ?it/s]

Negative biotic opportunity:   0%|          | 0/88 [00:00<?, ?it/s]

In [15]:
# Run this after Cell 8 (biotic opportunity matrices) in the enrichment
# notebook, before saving, to check how much real damage the 'unknown'
# exclusion does — i.e. how many species have SOME GloBI interaction
# records, but end up with zero usable (pos or neg) partners because all
# of their records happened to map to 'unknown'.
 
species_with_any_interaction = set()
net_clean_check = df_net.dropna(subset=['source_gbif_id', 'target_gbif_id'])
for row in net_clean_check.itertuples():
    src_idx = sp_to_idx.get(int(row.source_gbif_id)) if pd.notna(row.source_gbif_id) else None
    tgt_idx = sp_to_idx.get(int(row.target_gbif_id)) if pd.notna(row.target_gbif_id) else None
    if src_idx is not None:
        species_with_any_interaction.add(src_idx)
    if tgt_idx is not None:
        species_with_any_interaction.add(tgt_idx)
 
species_with_usable_partners = set(pos_partners.keys()) | set(neg_partners.keys())
stranded = species_with_any_interaction - species_with_usable_partners
 
print(f"Species with >=1 GloBI record       : {len(species_with_any_interaction):,}")
print(f"Species with >=1 usable pos/neg partner: {len(species_with_usable_partners):,}")
print(f"Stranded (records exist, all 'unknown'): {len(stranded):,}  "
      f"({len(stranded)/max(len(species_with_any_interaction),1):.1%} of species with any interaction)")

Species with >=1 GloBI record       : 163,841
Species with >=1 usable pos/neg partner: 146,139
Stranded (records exist, all 'unknown'): 17,702  (10.8% of species with any interaction)


In [23]:
# ==============================================================================
# 9. SAVE
# ==============================================================================
try:
    all_failed
except NameError:
    all_failed = []
    
total_pres = presence.sum().item()

print(f"\n💾 Saving to {output_path}...")
print(f"   Presence matrix  : {list(presence.shape)}  "
      f"({presence.nbytes / 1e6:.1f} MB)")
print(f"   Pos-opp matrix   : {list(pos_opp_mat.shape)}  "
      f"({pos_opp_mat.nbytes / 1e6:.1f} MB)")
print(f"   Neg-opp matrix   : {list(neg_opp_mat.shape)}  "
      f"({neg_opp_mat.nbytes / 1e6:.1f} MB)")
print(f"   Total presences  : {total_pres:,}  "
      f"(+{total_pres - n_seeded:,} over SINAS alone)")
print(f"   Species fetched  : "
      f"{len(completed) + len(new_completed):,} / {len(resolvable):,}")
if all_failed:
    print(f"   Permanently failed: {len(all_failed):,} species")

torch.save({
    'presence':           presence,         # (S, C) bool
    'pos_opp_mat':        pos_opp_mat,      # (S, C) float16 — positive-valence (resource/mutualism)
    'neg_opp_mat':        neg_opp_mat,      # (S, C) float16 — negative-valence (enemy/competition)
    'species_order':      all_species_ids,  # list[int] gbif_ids — same order as sp_to_idx
    'countries_order':    all_country_names,
    'complete':           len(all_failed) == 0,
    'permanently_failed': all_failed,
}, output_path)

print(f"✅ Saved.")

# Remove checkpoint only if fully complete with no failures
if len(all_failed) == 0 and checkpoint_path.exists():
    checkpoint_path.unlink()
    print("   Checkpoint removed.")
elif all_failed:
    print(f"   Checkpoint retained — re-run script to retry "
          f"{len(all_failed):,} permanently failed species.")


💾 Saving to c:\Users\simon\Documents\GitHub\horizon-scanner\data\gbif_presence.pt...
   Presence matrix  : [167357, 289]  (48.4 MB)
   Pos-opp matrix   : [167357, 289]  (96.7 MB)
   Neg-opp matrix   : [167357, 289]  (96.7 MB)
   Total presences  : 3,311,662  (+2,955,156 over SINAS alone)
   Species fetched  : 167,357 / 167,357
✅ Saved.
   Checkpoint removed.
